# Matrix Multiplication — PMPP Ch. 3 & 4

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!pip install nvcc4jupyter -q
%load_ext nvcc4jupyter

## Ch. 3 — Naive Matrix Multiplication

Each thread computes one output element `P[row][col]` by looping over a full row of M and column of N — every single value read comes from slow **global memory (DRAM)**.

For a 1000×1000 matrix: **2 billion global memory reads** (see Ch. 3 exercise).

In [ ]:
%%cuda
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define WIDTH 256

__global__ void matmul_naive(float *M, float *N, float *P, int width) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < width && col < width) {
        float sum = 0.0f;
        for (int k = 0; k < width; k++)
            sum += M[row * width + k] * N[k * width + col];
        P[row * width + col] = sum;
    }
}

int main() {
    size_t size = WIDTH * WIDTH * sizeof(float);

    float *h_M = (float*)malloc(size);
    float *h_N = (float*)malloc(size);
    float *h_P = (float*)malloc(size);

    // Identity x Identity = Identity (easy to verify)
    for (int i = 0; i < WIDTH; i++)
        for (int j = 0; j < WIDTH; j++) {
            h_M[i * WIDTH + j] = (i == j) ? 1.0f : 0.0f;
            h_N[i * WIDTH + j] = (i == j) ? 1.0f : 0.0f;
        }

    float *d_M, *d_N, *d_P;
    cudaMalloc(&d_M, size);
    cudaMalloc(&d_N, size);
    cudaMalloc(&d_P, size);

    cudaMemcpy(d_M, h_M, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_N, h_N, size, cudaMemcpyHostToDevice);

    dim3 block(16, 16);
    dim3 grid(WIDTH / 16, WIDTH / 16);
    matmul_naive<<<grid, block>>>(d_M, d_N, d_P, WIDTH);

    cudaMemcpy(h_P, d_P, size, cudaMemcpyDeviceToHost);

    int ok = 1;
    for (int i = 0; i < WIDTH && ok; i++)
        for (int j = 0; j < WIDTH && ok; j++) {
            float expected = (i == j) ? 1.0f : 0.0f;
            if (fabsf(h_P[i * WIDTH + j] - expected) > 1e-3f) ok = 0;
        }
    printf("Naive: %s\n", ok ? "correct" : "WRONG");
    printf("Total global memory reads: %d\n", WIDTH * WIDTH * WIDTH * 2);

    cudaFree(d_M); cudaFree(d_N); cudaFree(d_P);
    free(h_M); free(h_N); free(h_P);
    return 0;
}

## Ch. 4–5 — Tiled Matrix Multiplication with Shared Memory

Threads in a block collaboratively load a `TILE_WIDTH × TILE_WIDTH` tile of M and N into `__shared__` memory (fast on-chip SRAM), then compute from there.

- `__shared__` — on-chip memory, shared within a block, ~100× faster than global
- `__syncthreads()` — barrier: all threads must finish loading before any thread starts computing (Ch. 4.3)
- Each value reused `TILE_WIDTH` times → **16× fewer global reads** with `TILE_WIDTH=16`

In [ ]:
%%cuda
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define WIDTH      256
#define TILE_WIDTH 16

__global__ void matmul_tiled(float *M, float *N, float *P, int width) {
    __shared__ float Mds[TILE_WIDTH][TILE_WIDTH];
    __shared__ float Nds[TILE_WIDTH][TILE_WIDTH];

    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE_WIDTH + ty;
    int col = blockIdx.x * TILE_WIDTH + tx;

    float sum = 0.0f;

    for (int t = 0; t < width / TILE_WIDTH; t++) {
        // Collaboratively load tile into shared memory
        Mds[ty][tx] = M[row * width + (t * TILE_WIDTH + tx)];
        Nds[ty][tx] = N[(t * TILE_WIDTH + ty) * width + col];

        __syncthreads();

        for (int k = 0; k < TILE_WIDTH; k++)
            sum += Mds[ty][k] * Nds[k][tx];

        __syncthreads();
    }

    if (row < width && col < width)
        P[row * width + col] = sum;
}

int main() {
    size_t size = WIDTH * WIDTH * sizeof(float);

    float *h_M = (float*)malloc(size);
    float *h_N = (float*)malloc(size);
    float *h_P = (float*)malloc(size);

    for (int i = 0; i < WIDTH; i++)
        for (int j = 0; j < WIDTH; j++) {
            h_M[i * WIDTH + j] = (i == j) ? 1.0f : 0.0f;
            h_N[i * WIDTH + j] = (i == j) ? 1.0f : 0.0f;
        }

    float *d_M, *d_N, *d_P;
    cudaMalloc(&d_M, size);
    cudaMalloc(&d_N, size);
    cudaMalloc(&d_P, size);

    cudaMemcpy(d_M, h_M, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_N, h_N, size, cudaMemcpyHostToDevice);

    dim3 block(TILE_WIDTH, TILE_WIDTH);
    dim3 grid(WIDTH / TILE_WIDTH, WIDTH / TILE_WIDTH);
    matmul_tiled<<<grid, block>>>(d_M, d_N, d_P, WIDTH);

    cudaMemcpy(h_P, d_P, size, cudaMemcpyDeviceToHost);

    int ok = 1;
    for (int i = 0; i < WIDTH && ok; i++)
        for (int j = 0; j < WIDTH && ok; j++) {
            float expected = (i == j) ? 1.0f : 0.0f;
            if (fabsf(h_P[i * WIDTH + j] - expected) > 1e-3f) ok = 0;
        }
    printf("Tiled: %s\n", ok ? "correct" : "WRONG");
    printf("Global memory reads reduced by %dx vs naive\n", TILE_WIDTH);

    cudaFree(d_M); cudaFree(d_N); cudaFree(d_P);
    free(h_M); free(h_N); free(h_P);
    return 0;
}